# Heart Disease Prediction and Patient Risk Analysis
**Machine Learning Project | Google Colab / Jupyter Notebook**

This project follows the requested workflow:
1. Collect Data
2. Preprocess Data
3. Analyze Data
4. Build Linear Regression, Logistic Regression, KNN and K-Means models
5. Optimize and validate using cross-validation
6. Display prediction, risk score and risk category

**Dataset:** UCI Heart Disease (Cleveland subset), 303 records and 13 input features. The target is `num` (0 = no disease; 1–4 = disease present), converted to a binary target for prediction.

> **Important:** This is an educational machine-learning demonstration, not a medical diagnostic tool. Its score is a model output, not a clinically validated probability. Do not use it to make health decisions.

## 1. Install and import libraries

In [ ]:
!pip -q install ucimlrepo

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.base import clone

RANDOM_STATE = 42

## 2. Collect Data

In [ ]:
# Download the official UCI Heart Disease dataset (dataset ID 45)
heart = fetch_ucirepo(id=45)
X = heart.data.features.copy()
y_raw = heart.data.targets.iloc[:, 0].copy()

df = X.copy()
df["num"] = y_raw

# Standardize missing-value markers and convert values to numeric where possible
df = df.replace("?", np.nan)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values per column:")
display(df.isnull().sum())

## 3. Preprocess Data

In [ ]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", before - len(df))

# Target: 0 = no disease, 1 = disease present (original values 1,2,3,4)
df = df.dropna(subset=["num"]).copy()
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

X = df.drop(columns=["target"])
y = df["target"]

# Feature types in the Cleveland dataset
categorical_features = [c for c in ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"] if c in X.columns]
numeric_features = [c for c in X.columns if c not in categorical_features]

# Impute missing values; one-hot encode categorical variables; scale numeric values
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

### Outlier inspection (IQR method)

In [ ]:
# Flag potential outliers for analysis only; do not automatically delete clinical observations.
outlier_summary = {}
for col in numeric_features:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_summary[col] = int(((X_train[col] < low) | (X_train[col] > high)).sum())

display(pd.Series(outlier_summary, name="Potential outliers (IQR)"))

## 4. Analyze Data

In [ ]:
print("Target distribution:")
display(y.value_counts().rename(index={0:"No disease", 1:"Disease"}))

plt.figure(figsize=(5,4))
sns.countplot(x=y.map({0:"No disease", 1:"Disease"}))
plt.title("Heart Disease Class Distribution")
plt.xlabel("Target")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,7))
corr_df = df[numeric_features + ["target"]].corr(numeric_only=True)
sns.heatmap(corr_df, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

for col in [c for c in ["age", "trestbps", "chol", "thalach"] if c in df.columns]:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x="target", y=col)
    plt.title(f"{col} vs Heart Disease Target")
    plt.xlabel("Target (0 = No, 1 = Yes)")
    plt.tight_layout()
    plt.show()

## 5. Build Machine Learning Models

### 5A. Logistic Regression — Yes/No prediction
### 5B. KNN — similar-patient classification

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

knn_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

logistic_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
knn_pred = knn_model.predict(X_test)

def show_classification_results(name, y_true, y_pred):
    print(f"\\n{name}")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
    print("Recall   :", round(recall_score(y_true, y_pred, zero_division=0), 3))
    print("F1-score :", round(f1_score(y_true, y_pred, zero_division=0), 3))
    print(classification_report(y_true, y_pred, target_names=["No disease", "Disease"], zero_division=0))

show_classification_results("Logistic Regression", y_test, logistic_pred)
show_classification_results("KNN", y_test, knn_pred)

fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, name, pred in [(axes[0], "Logistic Regression", logistic_pred), (axes[1], "KNN", knn_pred)]:
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

### 5C. Linear Regression — estimated risk score (0–100)

In [ ]:
# Linear Regression is used only as the requested educational risk-score estimator.
linear_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", LinearRegression())
])
linear_model.fit(X_train, y_train)
linear_raw = linear_model.predict(X_test)
linear_score = np.clip(linear_raw, 0, 1) * 100

print("Linear Regression MAE:", round(mean_absolute_error(y_test, np.clip(linear_raw, 0, 1)), 3))
print("Linear Regression R2 :", round(r2_score(y_test, linear_raw), 3))
print("Example estimated scores (first 10):")
display(pd.DataFrame({"Actual target": y_test.iloc[:10].values, "Estimated risk score (%)": np.round(linear_score[:10], 1)}))

### 5D. K-Means Clustering — Low, Medium and High risk groups

In [ ]:
# K-Means is unsupervised. Cluster labels are ordered by each cluster's average disease-target rate.
kmeans_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=20))
])
kmeans_pipeline.fit(X_train)

train_clusters = kmeans_pipeline.predict(X_train)
cluster_risk_rate = pd.DataFrame({"cluster": train_clusters, "target": y_train.values}).groupby("cluster")["target"].mean()
ordered_clusters = cluster_risk_rate.sort_values().index.tolist()
risk_names = {ordered_clusters[0]:"Low", ordered_clusters[1]:"Medium", ordered_clusters[2]:"High"}

test_clusters = kmeans_pipeline.predict(X_test)
cluster_categories = [risk_names[c] for c in test_clusters]

display(pd.DataFrame({
    "Cluster": test_clusters,
    "Risk Category": cluster_categories
}).head(10))
print("Cluster disease rates (training data):")
display(cluster_risk_rate.sort_values())

## 6. Optimize the Model and Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Compare Logistic Regression and KNN using the same 5-fold stratified cross-validation
cv_log = cross_val_score(logistic_model, X_train, y_train, cv=cv, scoring="accuracy")
cv_knn = cross_val_score(knn_model, X_train, y_train, cv=cv, scoring="accuracy")

print("Logistic Regression CV accuracy:", np.round(cv_log, 3), "| Mean:", round(cv_log.mean(), 3))
print("KNN CV accuracy:", np.round(cv_knn, 3), "| Mean:", round(cv_knn.mean(), 3))

# Tune KNN's neighbour count using cross-validation
knn_grid = GridSearchCV(
    estimator=Pipeline(steps=[("preprocessor", clone(preprocessor)),
                              ("model", KNeighborsClassifier())]),
    param_grid={"model__n_neighbors":[3,5,7,9,11]},
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)
knn_grid.fit(X_train, y_train)
best_knn = knn_grid.best_estimator_
print("Best KNN parameters:", knn_grid.best_params_)
print("Best KNN CV accuracy:", round(knn_grid.best_score_, 3))
print("Tuned KNN test accuracy:", round(accuracy_score(y_test, best_knn.predict(X_test)), 3))

## 7. Display Final Outputs

For a new patient, enter values for the 13 input features. The code below uses one held-out test record as a demonstration. Replace it with a real input record only for classroom experimentation.

In [ ]:
# Demonstration patient: first row from the test set
sample_patient = X_test.iloc[[0]].copy()

# Logistic Regression: Yes/No prediction and model probability score
heart_prediction = int(logistic_model.predict(sample_patient)[0])
logistic_score = float(logistic_model.predict_proba(sample_patient)[0, 1] * 100)

# Linear Regression: requested separate risk score
linear_risk_score = float(np.clip(linear_model.predict(sample_patient)[0], 0, 1) * 100)

# K-Means: risk category based on cluster risk ordering
sample_cluster = int(kmeans_pipeline.predict(sample_patient)[0])
risk_category = risk_names[sample_cluster]

print("========== FINAL OUTPUT ==========")
print("Heart Disease Prediction:", "Yes" if heart_prediction == 1 else "No")
print("Logistic Regression model score:", round(logistic_score, 2), "%")
print("Linear Regression risk score:", round(linear_risk_score, 2), "%")
print("Patient Risk Category (K-Means):", risk_category)
display(sample_patient)

## 8. Predict for a manually entered patient (optional)

In [ ]:
# Edit these example values with values in the same units/categories as the UCI dataset.
# Categories: sex (0/1), cp (1-4), fbs (0/1), restecg (0-2),
# exang (0/1), slope (1-3), ca (0-3), thal (3/6/7).
new_patient = pd.DataFrame([{
    "age": 55, "sex": 1, "cp": 2, "trestbps": 130, "chol": 250,
    "fbs": 0, "restecg": 1, "thalach": 150, "exang": 0,
    "oldpeak": 1.0, "slope": 2, "ca": 0, "thal": 3
}])[X.columns]

new_prediction = int(logistic_model.predict(new_patient)[0])
new_logistic_score = float(logistic_model.predict_proba(new_patient)[0, 1] * 100)
new_linear_score = float(np.clip(linear_model.predict(new_patient)[0], 0, 1) * 100)
new_cluster = int(kmeans_pipeline.predict(new_patient)[0])

print("Prediction:", "Yes" if new_prediction else "No")
print("Logistic model score:", round(new_logistic_score, 2), "%")
print("Linear Regression risk score:", round(new_linear_score, 2), "%")
print("K-Means risk category:", risk_names[new_cluster])

## 9. Conclusion
The project collects the UCI Cleveland heart-disease data, preprocesses missing and duplicate values, explores relationships, and applies Linear Regression, Logistic Regression, KNN and K-Means. It evaluates classification models with a train/test split and 5-fold cross-validation, and displays a Yes/No prediction, model-generated score and Low/Medium/High cluster category.

**Limitations:** The dataset is small and historical. K-Means categories are relative to this dataset, and the Linear Regression score is not a clinical probability. Results must not be interpreted as diagnosis or treatment advice.

## Dataset citation
Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart Disease*. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X (CC BY 4.0).